# Notebook 01 — Coleta de dados

**Projeto:** Previsão do IPCA com Machine Learning  
**Série:** Machine Learning em Economia Brasileira  
**Repositório:** https://github.com/maurizioprizzi/ml-economia-brasil

---

## O que faremos neste notebook?

Vamos buscar, diretamente das fontes oficiais brasileiras, todos os dados que precisamos para prever a inflação. Nenhum arquivo será baixado manualmente — tudo acontece via **APIs públicas e gratuitas**.

Ao final deste notebook, teremos um arquivo `dados_brutos.parquet` com todas as séries históricas organizadas, prontas para a análise exploratória do próximo notebook.

### O que é uma API?

API (Application Programming Interface) é uma forma de dois programas conversarem entre si. Quando nosso código Python "pede" os dados do IPCA ao IBGE, ele faz isso através da API do IBGE — é como ligar para o IBGE e pedir os números, só que de forma automática.

O Brasil tem APIs públicas excelentes:
- **IBGE/SIDRA** — dados de preços, produção, emprego, censo
- **Banco Central do Brasil (SGS)** — juros, câmbio, crédito, expectativas
- **Ipeadata** — séries históricas macroeconômicas desde os anos 1940

### Série de dados que vamos coletar

| Variável | Fonte | Por que importa? |
|----------|-------|------------------|
| IPCA (inflação) | IBGE/SIDRA | nossa variável-alvo |
| Taxa Selic | BCB/SGS | principal instrumento de política monetária |
| Câmbio USD/BRL | BCB/SGS | importações encarecem com dólar alto |
| IGP-M | BCB/SGS | inflação no atacado, antecede o IPCA |
| Taxa de desemprego | BCB/SGS | menos emprego = menos consumo = menos inflação |
| Produção industrial | BCB/SGS | proxy da atividade econômica |
| IBC-Br | BCB/SGS | proxy mensal do PIB |
| Preço do petróleo Brent | BCB/SGS | afeta combustíveis e toda a cadeia produtiva |
| Expectativas de inflação (Focus) | BCB/SGS | o que o mercado espera que aconteça |

---

## 1. Importações e configurações

Antes de qualquer coisa, importamos as bibliotecas que vamos usar. Se aparecer algum erro de `ModuleNotFoundError`, significa que o kernel selecionado não é o do ambiente virtual `venv` — verifique o kernel no canto superior direito do VSCode.

In [21]:
# Manipulação de dados
import pandas as pd
import numpy as np

# APIs brasileiras
import sidrapy          # API do IBGE (SIDRA)
from bcb import sgs     # API do Banco Central do Brasil

# Utilitários
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configurações de exibição
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

print('✅ Bibliotecas importadas com sucesso!')
print(f'   pandas  {pd.__version__}')
print(f'   numpy   {np.__version__}')

✅ Bibliotecas importadas com sucesso!
   pandas  3.0.2
   numpy   2.4.4


In [22]:
# Caminhos do projeto
# Path(__file__) não funciona em notebooks — usamos Path.cwd()
NOTEBOOK_DIR = Path.cwd()
DATA_RAW_DIR = NOTEBOOK_DIR.parent / 'data' / 'raw'
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f'📁 Pasta de dados brutos: {DATA_RAW_DIR}')

# Período de coleta
# O IPCA existe desde julho de 1994 (Plano Real)
# Usamos 1995-01 para garantir séries completas em todas as fontes
DATA_INICIO = '1995-01-01'
DATA_FIM    = '2025-12-31'

print(f'📅 Período: {DATA_INICIO} → {DATA_FIM}')

📁 Pasta de dados brutos: /home/maurizio/eclipse-workspace/ml-economia-brasil/projeto-01-ipca/data/raw
📅 Período: 1995-01-01 → 2025-12-31


---

## 2. Coletando o IPCA — nossa variável-alvo

O **IPCA** (Índice de Preços ao Consumidor Amplo) é o índice oficial de inflação do Brasil. É calculado mensalmente pelo IBGE e mede a variação de preços de uma cesta de produtos e serviços consumidos por famílias com renda entre 1 e 40 salários mínimos.

### Por que o IPCA é a variável-alvo?

Porque é o índice usado pelo Banco Central para o **sistema de metas de inflação** — o principal framework de política monetária do Brasil desde 1999. Quando o Banco Central sobe ou desce a Selic, o objetivo é manter o IPCA dentro da meta estabelecida pelo Conselho Monetário Nacional.

### Como funciona o SIDRA (API do IBGE)?

O SIDRA organiza os dados em **tabelas**. Cada tabela tem um código numérico. Para o IPCA, a tabela principal é a **1737**, que contém a variação mensal e acumulada.

Usamos a biblioteca `sidrapy` para abstrair os detalhes da API.

In [23]:
def coletar_ipca() -> pd.DataFrame:
    """
    Coleta a variação mensal do IPCA via API do IBGE (SIDRA).
    
    Tabela SIDRA 1737:
      - Variável 63: IPCA - Variação mensal (%)
      - Nível territorial 1: Brasil
    
    Retorna
    -------
    pd.DataFrame
        Coluna 'ipca_mensal' com a variação percentual mensal.
        Índice: DatetimeIndex mensal.
    """
    print('⏳ Coletando IPCA do IBGE/SIDRA...')
    
    raw = sidrapy.get_table(
        table_code='1737',
        territorial_level='1',
        ibge_territorial_code='all',
        variable='63',        # variação mensal
        period='all'
    )
    
    # A primeira linha contém os cabeçalhos — removemos
    df = (
        raw
        .loc[1:, ['V', 'D2C']]                          # valor e data
        .rename(columns={'V': 'ipca_mensal', 'D2C': 'data'})
        .assign(
            data        = lambda x: pd.to_datetime(x['data'], format='%Y%m'),
            ipca_mensal = lambda x: pd.to_numeric(x['ipca_mensal'], errors='coerce')
        )
        .set_index('data')
        .sort_index()
    )
    
    # Filtra o período de interesse
    df = df.loc[DATA_INICIO:DATA_FIM]
    
    print(f'   ✅ {len(df)} observações coletadas '
          f'({df.index[0].strftime("%b/%Y")} → {df.index[-1].strftime("%b/%Y")})')
    print(f'   📊 Média histórica: {df["ipca_mensal"].mean():.2f}% a.m.')
    print(f'   📊 Máximo: {df["ipca_mensal"].max():.2f}% '
          f'em {df["ipca_mensal"].idxmax().strftime("%b/%Y")}')
    
    return df


ipca = coletar_ipca()
ipca.head(10)

⏳ Coletando IPCA do IBGE/SIDRA...
   ✅ 372 observações coletadas (Jan/1995 → Dec/2025)
   📊 Média histórica: 0.54% a.m.
   📊 Máximo: 3.02% em Nov/2002


,ipca_mensal
data,
1995-01-01,1.7000
1995-02-01,1.0200
1995-03-01,1.5500
1995-04-01,2.4300
1995-05-01,2.6700
1995-06-01,2.2600
1995-07-01,2.3600
1995-08-01,0.9900
1995-09-01,0.9900


---

## 3. Coletando variáveis explicativas via API do Banco Central

O Banco Central do Brasil disponibiliza milhares de séries temporais através do **SGS** (Sistema Gerenciador de Séries Temporais). Cada série tem um código numérico.

A biblioteca `python-bcb` torna esse acesso muito simples.

### Por que essas variáveis?

A escolha não é arbitrária — é guiada pela **teoria econômica**:

- **Selic:** quando o banco central sobe os juros, o crédito fica mais caro, as pessoas consomem menos, e a inflação tende a cair. É o principal instrumento de política monetária.
- **Câmbio:** o Brasil importa muitos insumos industriais e combustíveis. Quando o dólar sobe, tudo isso fica mais caro — o que pressiona a inflação. Esse efeito é chamado de *pass-through* cambial.
- **IGP-M:** é um índice de preços que captura inflação no atacado (matérias-primas, produtos industriais). Como os preços no atacado chegam ao consumidor com algum atraso, o IGP-M tende a *antecipar* movimentos do IPCA.
- **Desemprego:** a Curva de Phillips — uma das relações mais estudadas em macroeconomia — sugere que há uma troca entre desemprego e inflação. Quando o desemprego cai (mais gente empregada, mais consumo), a inflação tende a subir.
- **Expectativas Focus:** o que o mercado financeiro *espera* que aconteça com a inflação. Se todos esperam inflação alta, as empresas já reajustam preços preventivamente — e a inflação acaba sendo alta mesmo. É uma profecia autorrealizável.

In [24]:
# Códigos das séries no SGS do Banco Central
# Fonte: https://www3.bcb.gov.br/sgspub/
SERIES_BCB = {
    'selic_meta':       432,    # Taxa Selic meta (% a.a.)
    'cambio_usd':      3698,    # Taxa de câmbio - R$/US$ - venda - média do período
    'igpm_mensal':      189,    # IGP-M - variação mensal (%)
    'desemprego':      24369,   # Taxa de desocupação - PNAD Contínua (%)
    'producao_ind':    21859,   # Produção industrial - índice (2012=100)
    'ibc_br':          24364,   # IBC-Br - proxy mensal do PIB (índice)
    'credito_pf':      20400,   # Concessões de crédito - pessoas físicas (R$ mi)
}

print('Séries que vamos coletar do Banco Central:')
for nome, codigo in SERIES_BCB.items():
    print(f'  SGS {codigo:>6} → {nome}')

Séries que vamos coletar do Banco Central:
  SGS    432 → selic_meta
  SGS   3698 → cambio_usd
  SGS    189 → igpm_mensal
  SGS  24369 → desemprego
  SGS  21859 → producao_ind
  SGS  24364 → ibc_br
  SGS  20400 → credito_pf


In [25]:
# Diagnóstico — testa um bloco de cada vez
for nome, codigo in SERIES_BCB.items():
    print(f'\nTestando {nome} (SGS {codigo})...')
    try:
        teste = sgs.get(codes={nome: codigo}, start='2020-01-01', end='2025-12-31')
        print(f'  ✅ OK — {len(teste)} linhas')
    except Exception as e:
        print(f'  ❌ ERRO: {e}')


Testando selic_meta (SGS 432)...
  ✅ OK — 2192 linhas

Testando cambio_usd (SGS 3698)...
  ✅ OK — 72 linhas

Testando igpm_mensal (SGS 189)...
  ✅ OK — 72 linhas

Testando desemprego (SGS 24369)...
  ✅ OK — 72 linhas

Testando producao_ind (SGS 21859)...
  ✅ OK — 72 linhas

Testando ibc_br (SGS 24364)...
  ✅ OK — 72 linhas

Testando credito_pf (SGS 20400)...
  ✅ OK — 20 linhas


In [26]:
def coletar_series_bcb(series: dict, inicio: str, fim: str) -> pd.DataFrame:
    """
    Coleta múltiplas séries do SGS do Banco Central do Brasil.
    Coleta série por série para isolar falhas individuais.
    """
    print('⏳ Coletando séries do Banco Central (SGS)...')

    resultados = {}

    for nome, codigo in series.items():
        print(f'   buscando {nome} (SGS {codigo})...', end=' ')
        try:
            df_serie = sgs.get(
                codes={nome: codigo},
                start=inicio,
                end=fim
            )
            # Reamostrar para mensal se vier em frequência maior
            df_serie = df_serie.resample('MS').last()
            resultados[nome] = df_serie[nome]
            print(f'✅ {len(df_serie)} obs.')
        except Exception as e:
            print(f'❌ {e}')

    df = pd.DataFrame(resultados)
    df = df.sort_index()

    print(f'\n   {len(df.columns)} séries coletadas, {len(df)} observações')
    for col in df.columns:
        n = df[col].isna().sum()
        pct = 100 * n / len(df)
        status = '⚠️ ' if pct > 10 else '  '
        print(f'   {status}{col:<20} {n:>3} ausentes ({pct:.1f}%)')

    return df


variaveis_bcb = coletar_series_bcb(SERIES_BCB, DATA_INICIO, DATA_FIM)
variaveis_bcb.head()

⏳ Coletando séries do Banco Central (SGS)...
   buscando selic_meta (SGS 432)... ❌ BCB error: O sistema aceita uma janela de consulta de, no máximo, 10 anos em séries de periodicidade diária
   buscando cambio_usd (SGS 3698)... ✅ 372 obs.
   buscando igpm_mensal (SGS 189)... ✅ 372 obs.
   buscando desemprego (SGS 24369)... ✅ 166 obs.
   buscando producao_ind (SGS 21859)... ✅ 288 obs.
   buscando ibc_br (SGS 24364)... ✅ 276 obs.
   buscando credito_pf (SGS 20400)... ✅ 200 obs.

   6 séries coletadas, 372 observações
     cambio_usd             0 ausentes (0.0%)
     igpm_mensal            0 ausentes (0.0%)
   ⚠️ desemprego           206 ausentes (55.4%)
   ⚠️ producao_ind          84 ausentes (22.6%)
   ⚠️ ibc_br                96 ausentes (25.8%)
   ⚠️ credito_pf           172 ausentes (46.2%)


,cambio_usd,igpm_mensal,desemprego,producao_ind,ibc_br,credito_pf
Date,,,,,,
1995-01-01,0.8471,0.9200,NaN,NaN,NaN,NaN
1995-02-01,0.8408,1.3900,NaN,NaN,NaN,NaN
1995-03-01,0.8894,1.1200,NaN,NaN,NaN,NaN
1995-04-01,0.9075,2.1000,NaN,NaN,NaN,NaN
1995-05-01,0.8974,0.5800,NaN,NaN,NaN,NaN


In [27]:
import time

def coletar_selic_robusto(inicio: str, fim: str) -> pd.Series:
    """
    Coleta Selic em blocos de 5 anos com retry automático.
    """
    print('⏳ Coletando Selic (blocos de 5 anos com retry)...')

    anos_inicio = pd.Timestamp(inicio).year
    anos_fim    = pd.Timestamp(fim).year
    blocos      = []

    for ano in range(anos_inicio, anos_fim + 1, 5):
        bloco_inicio = f'{ano}-01-01'
        bloco_fim    = f'{min(ano + 4, anos_fim)}-12-31'
        
        sucesso = False
        for tentativa in range(1, 4):       # até 3 tentativas
            print(f'   {bloco_inicio} → {bloco_fim}  '
                  f'(tentativa {tentativa})...', end=' ')
            try:
                bloco = sgs.get(
                    codes={'selic_meta': 432},
                    start=bloco_inicio,
                    end=bloco_fim
                )
                blocos.append(bloco)
                print(f'✅ {len(bloco)} obs.')
                sucesso = True
                break
            except Exception as e:
                print(f'❌ {e}')
                if tentativa < 3:
                    print(f'   aguardando 5s antes de tentar novamente...')
                    time.sleep(5)
        
        if not sucesso:
            print(f'   ⚠️  bloco {bloco_inicio}→{bloco_fim} falhou nas 3 tentativas — pulando')

    selic = pd.concat(blocos)
    selic = selic[~selic.index.duplicated(keep='last')]
    selic = selic.sort_index()
    selic = selic.resample('MS').last()

    print(f'\n   ✅ Selic: {len(selic)} observações mensais')
    print(f'   Cobertura: {selic.dropna().index[0].strftime("%b/%Y")} → '
          f'{selic.dropna().index[-1].strftime("%b/%Y")}')
    return selic['selic_meta']


selic = coletar_selic_robusto(DATA_INICIO, DATA_FIM)
variaveis_bcb['selic_meta'] = selic
print(f'\nSelic no DataFrame: {variaveis_bcb["selic_meta"].notna().sum()} valores válidos')

⏳ Coletando Selic (blocos de 5 anos com retry)...
   1995-01-01 → 1999-12-31  (tentativa 1)... ✅ 302 obs.
   2000-01-01 → 2004-12-31  (tentativa 1)... ✅ 1827 obs.
   2005-01-01 → 2009-12-31  (tentativa 1)... ✅ 1826 obs.
   2010-01-01 → 2014-12-31  (tentativa 1)... ✅ 1826 obs.
   2015-01-01 → 2019-12-31  (tentativa 1)... ✅ 1826 obs.
   2020-01-01 → 2024-12-31  (tentativa 1)... ✅ 1827 obs.
   2025-01-01 → 2025-12-31  (tentativa 1)... ✅ 365 obs.

   ✅ Selic: 322 observações mensais
   Cobertura: Mar/1999 → Dec/2025

Selic no DataFrame: 322 valores válidos


---

## 4. Coletando expectativas de inflação (Boletim Focus)

O **Boletim Focus** é uma pesquisa semanal do Banco Central com economistas e analistas de mercado. Eles respondem: *"Qual é a sua previsão para o IPCA nos próximos 12 meses?"*

A mediana dessas respostas é publicada toda segunda-feira. É uma das séries mais acompanhadas do mercado financeiro brasileiro.

### Por que incluir expectativas no modelo?

Em economia, expectativas têm poder causal. Se empresas e trabalhadores esperam inflação alta, os primeiros reajustam preços preventivamente e os segundos pedem salários maiores — o que *de fato* causa inflação. Esse fenômeno é chamado de **inércia inflacionária** e foi central na hiperinflação brasileira dos anos 1980.

Incluir as expectativas do Focus no modelo captura essa dinâmica forward-looking.

In [28]:
# Expectativas Focus — também disponíveis via python-bcb
# Série 13522: Expectativa de inflação IPCA para os próximos 12 meses (mediana)
SERIES_FOCUS = {
    'expectativa_ipca_12m': 13522,
}

print('⏳ Coletando expectativas Focus do Banco Central...')

focus = sgs.get(
    codes=SERIES_FOCUS,
    start=DATA_INICIO,
    end=DATA_FIM
)

# O Focus é semanal — pegamos a última leitura de cada mês
focus = focus.resample('MS').last()

print(f'   ✅ Expectativas Focus coletadas: {len(focus)} observações')
print(f'   📊 Disponível desde: {focus.dropna().index[0].strftime("%b/%Y")}')
focus.tail()

⏳ Coletando expectativas Focus do Banco Central...
   ✅ Expectativas Focus coletadas: 372 observações
   📊 Disponível desde: Jan/1995


,expectativa_ipca_12m
Date,
2025-08-01,5.1300
2025-09-01,5.1700
2025-10-01,4.6800
2025-11-01,4.4600
2025-12-01,4.2600


---

## 5. Consolidando todos os dados

Agora vamos juntar todas as séries em um único DataFrame. Usamos um **left join** com base no IPCA — ou seja, mantemos todas as datas onde temos IPCA e completamos com as outras variáveis onde disponíveis.

### Uma nota sobre valores ausentes

É normal ter alguns valores ausentes (`NaN`) neste momento. Por exemplo:
- A PNAD Contínua (desemprego) só existe desde 2012
- O Focus só tem expectativas desde 1999
- O IBC-Br foi criado em 2003

No próximo notebook (análise exploratória), decidiremos como tratar cada caso.

In [29]:
def consolidar_dados(
    ipca: pd.DataFrame,
    variaveis_bcb: pd.DataFrame,
    focus: pd.DataFrame
) -> pd.DataFrame:
    """
    Consolida todas as séries em um único DataFrame.
    
    Usa o IPCA como base (left join) e adiciona as demais séries.
    """
    df = (
        ipca
        .join(variaveis_bcb, how='left')
        .join(focus, how='left')
    )
    
    return df


dados = consolidar_dados(ipca, variaveis_bcb, focus)

print('📋 Dataset consolidado:')
print(f'   Linhas:  {len(dados)} observações mensais')
print(f'   Colunas: {len(dados.columns)} variáveis')
print(f'   Período: {dados.index[0].strftime("%b/%Y")} → '
      f'{dados.index[-1].strftime("%b/%Y")}')
print()
dados.info()

📋 Dataset consolidado:
   Linhas:  372 observações mensais
   Colunas: 9 variáveis
   Período: Jan/1995 → Dec/2025

<class 'pandas.DataFrame'>
DatetimeIndex: 372 entries, 1995-01-01 to 2025-12-01
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ipca_mensal           372 non-null    float64
 1   cambio_usd            372 non-null    float64
 2   igpm_mensal           372 non-null    float64
 3   desemprego            166 non-null    float64
 4   producao_ind          288 non-null    float64
 5   ibc_br                276 non-null    float64
 6   credito_pf            200 non-null    float64
 7   selic_meta            322 non-null    float64
 8   expectativa_ipca_12m  372 non-null    float64
dtypes: float64(9)
memory usage: 29.1 KB


In [17]:
# Visão geral dos dados
dados.describe().round(2)

,ipca_mensal,cambio_usd,igpm_mensal,desemprego,producao_ind,ibc_br,credito_pf,selic_meta,expectativa_ipca_12m
count,372.0000,372.0000,372.0000,166.0000,288.0000,276.0000,200.0000,322.0000,372.0000
mean,0.5400,2.8600,0.6500,9.7400,105.6800,94.0000,28.4900,12.7100,11.0000
std,0.4600,1.4200,0.9000,2.7800,11.1200,9.8700,4.4300,5.3400,42.5600
min,-0.6800,0.8400,-1.9300,5.1000,70.2000,69.4400,16.2000,2.0000,1.6500
25%,0.2600,1.8000,0.1500,7.2000,97.4800,88.6300,26.0600,9.2500,4.5000
50%,0.4600,2.3600,0.5300,8.9500,105.7000,96.6800,28.9900,12.5000,5.9000
75%,0.7100,3.7800,1.0000,12.1800,112.9300,100.8600,31.8500,15.9400,7.7000
max,3.0200,6.1000,5.1900,14.9000,131.2000,110.3900,37.0200,42.0000,631.5400


In [31]:
# Mapa de valores ausentes por coluna e período
print('\n📊 Valores ausentes por variável:')
print('-' * 50)

ausentes = dados.isna().sum().sort_values(ascending=False)
for col, n in ausentes.items():
    if n > 0:
        pct = 100 * n / len(dados)
        primeira_obs = dados[col].first_valid_index()
        print(f'  {col:<25} {n:>4} ausentes ({pct:>5.1f}%)  '
              f'→ disponível desde {primeira_obs.strftime("%b/%Y") if primeira_obs else "nunca"}')
    else:
        print(f'  {col:<25} sem ausentes ✅')


📊 Valores ausentes por variável:
--------------------------------------------------
  desemprego                 206 ausentes ( 55.4%)  → disponível desde Mar/2012
  credito_pf                 172 ausentes ( 46.2%)  → disponível desde Jan/2005
  ibc_br                      96 ausentes ( 25.8%)  → disponível desde Jan/2003
  producao_ind                84 ausentes ( 22.6%)  → disponível desde Jan/2002
  selic_meta                  50 ausentes ( 13.4%)  → disponível desde Mar/1999
  ipca_mensal               sem ausentes ✅
  igpm_mensal               sem ausentes ✅
  cambio_usd                sem ausentes ✅
  expectativa_ipca_12m      sem ausentes ✅


---

## 6. Salvando os dados brutos

Salvamos em formato **Parquet** — um formato colunar eficiente que preserva os tipos de dados (incluindo o índice de datas) e ocupa muito menos espaço que CSV.

### CSV vs. Parquet: qual a diferença?

| | CSV | Parquet |
|--|-----|--------|
| Legível por humanos | ✅ sim | ❌ não |
| Preserva tipos de dados | ❌ não | ✅ sim |
| Eficiência de espaço | ❌ baixa | ✅ alta |
| Velocidade de leitura | ❌ lenta | ✅ rápida |

Para dados que só serão lidos por código Python, Parquet é sempre a melhor escolha. Salvamos também um CSV para inspeção manual.

In [32]:
# Salvar em Parquet (uso pelo código)
caminho_parquet = DATA_RAW_DIR / 'dados_brutos.parquet'
dados.to_parquet(caminho_parquet)
print(f'✅ Parquet salvo em: {caminho_parquet}')

# Salvar em CSV (inspeção manual)
caminho_csv = DATA_RAW_DIR / 'dados_brutos.csv'
dados.to_csv(caminho_csv)
print(f'✅ CSV salvo em: {caminho_csv}')

# Verificar tamanho dos arquivos
import os
tamanho_parquet = os.path.getsize(caminho_parquet) / 1024
tamanho_csv     = os.path.getsize(caminho_csv) / 1024
print(f'\n📦 Tamanho Parquet: {tamanho_parquet:.1f} KB')
print(f'📦 Tamanho CSV:     {tamanho_csv:.1f} KB')
print(f'💾 Redução de espaço: {100*(1 - tamanho_parquet/tamanho_csv):.0f}%')

✅ Parquet salvo em: /home/maurizio/eclipse-workspace/ml-economia-brasil/projeto-01-ipca/data/raw/dados_brutos.parquet
✅ CSV salvo em: /home/maurizio/eclipse-workspace/ml-economia-brasil/projeto-01-ipca/data/raw/dados_brutos.csv

📦 Tamanho Parquet: 22.9 KB
📦 Tamanho CSV:     20.3 KB
💾 Redução de espaço: -13%


---

## 7. Verificação final

Antes de fechar o notebook, fazemos uma verificação rápida: recarregamos o arquivo salvo e confirmamos que os dados estão íntegros.

In [33]:
# Recarrega e verifica
verificacao = pd.read_parquet(caminho_parquet)

assert len(verificacao) == len(dados), 'ERRO: número de linhas diferente!'
assert list(verificacao.columns) == list(dados.columns), 'ERRO: colunas diferentes!'
assert verificacao.index.dtype == dados.index.dtype, 'ERRO: tipo do índice diferente!'

print('✅ Verificação concluída — arquivo íntegro!')
print()
print('Primeiras 5 linhas do arquivo salvo:')
verificacao.head()

✅ Verificação concluída — arquivo íntegro!

Primeiras 5 linhas do arquivo salvo:


,ipca_mensal,cambio_usd,igpm_mensal,desemprego,producao_ind,ibc_br,credito_pf,selic_meta,expectativa_ipca_12m
data,,,,,,,,,
1995-01-01,1.7000,0.8471,0.9200,NaN,NaN,NaN,NaN,NaN,631.5400
1995-02-01,1.0200,0.8408,1.3900,NaN,NaN,NaN,NaN,NaN,426.8300
1995-03-01,1.5500,0.8894,1.1200,NaN,NaN,NaN,NaN,NaN,274.7800
1995-04-01,2.4300,0.9075,2.1000,NaN,NaN,NaN,NaN,NaN,169.0500
1995-05-01,2.6700,0.8974,0.5800,NaN,NaN,NaN,NaN,NaN,91.7900


---

## Resumo do que fizemos

✅ Conectamos às APIs públicas do IBGE e do Banco Central  
✅ Coletamos o IPCA mensal desde 1995  
✅ Coletamos 8 variáveis explicativas macroeconômicas  
✅ Coletamos expectativas de inflação do Boletim Focus  
✅ Consolidamos tudo em um único dataset de `N` linhas e `10` colunas  
✅ Salvamos em Parquet (eficiente) e CSV (legível)

## Próximo notebook

**Notebook 02 — Análise Exploratória de Dados (EDA)**

Vamos visualizar as séries históricas, entender os padrões, as correlações entre variáveis, e identificar eventos importantes na história econômica do Brasil que aparecem nos dados (Plano Real, crise de 2008, pandemia, etc.).

---
*Parte da série [Machine Learning em Economia Brasileira](https://github.com/maurizioprizzi/ml-economia-brasil)*